##  SHAP Analysis with Single-Feature XGBoost Models

In this analysis, we aim to understand the **individual impact of each continuous input feature** on three wildfire-related target variables:

- `surface_consumed`
- `midstory_consumed`
- `canopy_consumed`

###  Methodology

For each `(target, feature)` pair:
1. **Data Cleaning**: We remove any rows where either the feature or the target has missing (`NaN`) or infinite values.
2. **Train/Test Split**: We split the cleaned data using `train_test_split()` with a fixed random seed for reproducibility.
3. **Model Training**: We fit a **single-feature XGBoost regression model** using:
   - 100 estimators
   - max depth of 4
   - learning rate of 0.1
4. **Model Evaluation**:
   - We compute the **R² score** to measure how well the feature alone predicts the target.
5. **SHAP Analysis**:
   - We use SHAP (SHapley Additive exPlanations) to compute the **mean absolute SHAP value** for the feature, representing its **average contribution** to the model's predictions.

###  Output

For each `(target, feature)` combination, we collect:
- `r2_score`: Model's predictive performance
- `mean_shap`: Feature's average influence on predictions

These results help us **rank features by importance** and interpret their **predictive value individually**.



In [24]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
import shap

# Set random seed
seed=np.random.randint(0,10000)

# Define feature and target variables
continuous_features = ["gap_length", "wind_speed", "spacing_distance", "igniter_velocity", "lfm", "fdfm", "wind_direction"]
target_vars = ["surface_consumed", "midstory_consumed", "canopy_consumed"]

# Load the dataframe
df = pd.read_csv('cleaned_df.csv')
results = []

# Loop over each target variable
for target in target_vars:
    y = df[target]

    # Loop over each feature
    for feature in continuous_features:
        X = df[[feature]]

        # Clean the data
        Xy = pd.concat([X, y], axis=1)
        Xy = Xy.replace([np.inf, -np.inf], np.nan).dropna()
        X = Xy[[feature]]
        y = Xy[target]

        # Train/test split
        x_train, x_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=seed
        )

        # Train model
        model = XGBRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=seed
        )
        model.fit(x_train, y_train)

        # Evaluate R²
        y_pred = model.predict(x_test)
        score = r2_score(y_test, y_pred)

        # Compute SHAP values
        explainer = shap.Explainer(model)
        shap_values = explainer(x_test)
        mean_shap = np.abs(shap_values.values).mean()

        # Save results
        results.append({
            "target": target,
            "feature": feature,
            "r2_score": score,
            "mean_shap": mean_shap
        })

In [25]:
# Convert to DataFrame and sort
results_df = pd.DataFrame(results)

In [26]:
#Sort by mean shap
print(results_df.sort_values(by=["target", "mean_shap"], ascending=[True, False]).to_string(index=False))
#Note: Sorting happens per target output variable.

           target          feature  r2_score  mean_shap
  canopy_consumed   wind_direction  0.300621   0.033547
  canopy_consumed igniter_velocity  0.223767   0.028266
  canopy_consumed       gap_length  0.227235   0.025659
  canopy_consumed       wind_speed  0.186201   0.024995
  canopy_consumed             fdfm  0.139710   0.018156
  canopy_consumed              lfm  0.104401   0.017674
  canopy_consumed spacing_distance  0.163557   0.016836
midstory_consumed       gap_length  0.258815   0.087981
midstory_consumed   wind_direction  0.226703   0.084282
midstory_consumed       wind_speed  0.201429   0.080337
midstory_consumed             fdfm  0.125954   0.066127
midstory_consumed igniter_velocity  0.175359   0.064040
midstory_consumed spacing_distance  0.113292   0.058160
midstory_consumed              lfm  0.104642   0.046861
 surface_consumed       gap_length  0.379577   0.090100
 surface_consumed   wind_direction  0.187426   0.087525
 surface_consumed igniter_velocity  0.318454   0

In [27]:
#Sort by R2 score
print(results_df.sort_values(by=["target", "r2_score"], ascending=[True, False]).to_string(index=False))
#Note: Sorting happens per target output variable.

           target          feature  r2_score  mean_shap
  canopy_consumed   wind_direction  0.300621   0.033547
  canopy_consumed       gap_length  0.227235   0.025659
  canopy_consumed igniter_velocity  0.223767   0.028266
  canopy_consumed       wind_speed  0.186201   0.024995
  canopy_consumed spacing_distance  0.163557   0.016836
  canopy_consumed             fdfm  0.139710   0.018156
  canopy_consumed              lfm  0.104401   0.017674
midstory_consumed       gap_length  0.258815   0.087981
midstory_consumed   wind_direction  0.226703   0.084282
midstory_consumed       wind_speed  0.201429   0.080337
midstory_consumed igniter_velocity  0.175359   0.064040
midstory_consumed             fdfm  0.125954   0.066127
midstory_consumed spacing_distance  0.113292   0.058160
midstory_consumed              lfm  0.104642   0.046861
 surface_consumed       gap_length  0.379577   0.090100
 surface_consumed igniter_velocity  0.318454   0.080944
 surface_consumed   wind_direction  0.187426   0

I added R2 score as this tells us how well the model predicts. An R2 score of 1.0 is a perfect prediction. Taking this into consideration, the R2 scores are a bit low...